<a href="https://colab.research.google.com/github/alhakimia542-ctrl/NLP-projects/blob/main/BETR_CLOP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os
import re
from google.colab import drive

# 1. ربط الدرايف
drive.mount('/content/drive')

# 2. دالة ذكية للبحث عن المسار الحقيقي للملف داخل الدرايف
def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

print("🔎 جاري البحث عن ملف UltimateArabicPrePos.csv في الدرايف... قد يستغرق ذلك دقيقة...")
file_path = find_file('UltimateArabicPrePos.csv', '/content/drive/MyDrive/')

if file_path:
    print(f"✅ تم العثور على الملف في: {file_path}")
    df = pd.read_csv(file_path)

    # --- الآن نبدأ المعالجة المسبقة الشاملة بعد التأكد من وجود df ---

    # أ. تنظيف الشوائب والقيم الفارغة والمكررة
    df.dropna(subset=['text', 'label'], inplace=True)
    df.drop_duplicates(subset=['text'], inplace=True)

    def clean_arabic(text):
        if not isinstance(text, str): return ""
        text = re.sub(r'http\S+|www\S+|https\S+', '', text) # روابط
        text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text) # تشكيل
        text = re.sub(r'[^\u0600-\u06ff\s]', ' ', text) # رموز
        return re.sub(r'\s+', ' ', text).strip()

    print("🧹 جاري التنظيف العميق والموازنة...")
    df['text'] = df['text'].apply(clean_arabic)
    df = df[df['text'].str.len() > 15] # حذف النصوص القصيرة

    # ب. موازنة البيانات (7000 لكل فئة)
    df_balanced = df.groupby('label').apply(
        lambda x: x.sample(n=min(len(x), 7000), random_state=42)
    ).reset_index(drop=True)

    # ج. حفظ الملف الجاهز في نفس مجلد الملف الأصلي
    save_dir = os.path.dirname(file_path)
    output_file = os.path.join(save_dir, 'Ultimate_Ready_For_BERT.csv')
    df_balanced.to_csv(output_file, index=False)

    print(f"🎉 نجحت العملية! الحجم النهائي: {len(df_balanced)}")
    print(f"💾 تم حفظ الملف المنظف هنا: {output_file}")

else:
    print("❌ للأسف لم أجد الملف. تأكد من رفعه أو فك الضغط عنه داخل Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔎 جاري البحث عن ملف UltimateArabicPrePos.csv في الدرايف... قد يستغرق ذلك دقيقة...
✅ تم العثور على الملف في: /content/drive/MyDrive/Ultimate_Extracted/UltimateArabicPrePos.csv
🧹 جاري التنظيف العميق والموازنة...


/tmp/ipython-input-366437469.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('label').apply(


🎉 نجحت العملية! الحجم النهائي: 63338
💾 تم حفظ الملف المنظف هنا: /content/drive/MyDrive/Ultimate_Extracted/Ultimate_Ready_For_BERT.csv


In [ ]:
import pandas as pd
import torch
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# 1. تحميل البيانات الجاهزة من المسار الذي وجده الكود السابق
ready_file_path = '/content/drive/MyDrive/Ultimate_Extracted/Ultimate_Ready_For_BERT.csv'
df = pd.read_csv(ready_file_path)

# 2. تحويل التصنيفات إلى أرقام
le = LabelEncoder()
df['label_idx'] = le.fit_transform(df['label'])
num_labels = len(le.classes_)

# 3. تحميل الـ Tokenizer الخاص بـ AraBERT
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4. تقسيم البيانات وتجهيز التنسيق لـ PyTorch
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].astype(str).tolist(), df['label_idx'].tolist(), test_size=0.15, random_state=42
)

class ArabicDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self): return len(self.labels)

train_dataset = ArabicDataset(train_texts, train_labels, tokenizer)
val_dataset = ArabicDataset(val_texts, val_labels, tokenizer)

# 5. تحميل النموذج
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# 6. إعدادات التدريب (مع التعديل المصلح للخطأ)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",           # التعديل الصحيح هنا
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                       # تسريع باستخدام GPU
    logging_steps=100
)

# 7. بدء عملية التدريب
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("🚀 جاري بدء تدريب AraBERT الآن... راقب شريط التقدم بالأسفل")
trainer.train()

# 8. حفظ النموذج النهائي في الدرايف
save_path = "/content/drive/MyDrive/Ultimate_Extracted/AraBERT_Model_Final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# حفظ الـ LabelEncoder
with open(f'{save_path}/label_encoder_bert.pickle', 'wb') as f:
    pickle.dump(le, f)

print(f"✅ مبروك! انتهى التدريب وتم حفظ النموذج في: {save_path}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 جاري بدء تدريب AraBERT الآن... راقب شريط التقدم بالأسفل


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.177600,0.196919
2,0.091500,0.177509
3,0.098200,0.193040


✅ مبروك! انتهى التدريب وتم حفظ النموذج في: /content/drive/MyDrive/Ultimate_Extracted/AraBERT_Model_Final


In [ ]:
# 1. قائمة النصوص التي تريد اختبارها (أضف ما شئت من الجمل هنا)
test_sentences = [
    "سجل اللاعب هدفاً رائعاً في الدقيقة الأخيرة من المباراة",
    "ارتفاع أسعار النفط عالمياً يؤثر على ميزانيات الدول الناشئة",
    "اكتشاف طبي جديد يساعد في علاج أمراض القلب دون جراحة",
    "انطلاق فعاليات المهرجان الثقافي السنوي في وسط المدينة",
    "البورصة تشهد تراجعاً ملحوظاً في تداولات اليوم"
]

print(f"🔎 جاري تحليل {len(test_sentences)} نصوص...\n")

# 2. تنفيذ التنبؤ وعرض النتائج في جدول أنيق
results = []
for text in test_sentences:
    category = predict_category(text) # نستخدم الدالة التي عرفناها في الكود السابق
    results.append({"النص": text, "التصنيف المتوقع": category})

# عرض النتائج
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

🔎 جاري تحليل 5 نصوص...



,النص,التصنيف المتوقع
0,سجل اللاعب هدفاً رائعاً في الدقيقة الأخيرة من ...,Sport
1,ارتفاع أسعار النفط عالمياً يؤثر على ميزانيات ا...,Economy
2,اكتشاف طبي جديد يساعد في علاج أمراض القلب دون ...,Medical
3,انطلاق فعاليات المهرجان الثقافي السنوي في وسط ...,Culture
4,البورصة تشهد تراجعاً ملحوظاً في تداولات اليوم,Economy


In [ ]:
import shutil
from google.colab import files

# 1. تحديد المسار الذي حفظنا فيه النموذج في الدرايف
model_path = '/content/drive/MyDrive/Ultimate_Extracted/AraBERT_Model_Final'

# 2. اسم الملف المضغوط الذي سيظهر على جهازك
zip_name = 'My_Arabic_BERT_Model'

print("⏳ جاري ضغط ملفات النموذج (حوالي 500 ميجابايت)... انتظر قليلاً")

# 3. عملية الضغط
shutil.make_archive(zip_name, 'zip', model_path)

print(f"✅ تمت عملية الضغط بنجاح.")
print("🚀 سيبدأ التحميل الآن إلى جهازك الشخصي...")

# 4. أمر التحميل المباشر للمتصفح
files.download(f'{zip_name}.zip')

⏳ جاري ضغط ملفات النموذج (حوالي 500 ميجابايت)... انتظر قليلاً
✅ تمت عملية الضغط بنجاح.
🚀 سيبدأ التحميل الآن إلى جهازك الشخصي...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>